# Li-ion State-of-Charge Estimation: Coulomb Counting vs. Extended Kalman Filter

Run the whole project in your browser. Nothing is installed on your machine and
no account is needed beyond a Google login.

Press **Runtime -> Run all**, or run the cells one at a time.

Repository: https://github.com/fatinnihal532-hub/liion-soc-ekf


## Setup


In [ ]:
!git clone -q https://github.com/fatinnihal532-hub/liion-soc-ekf.git
%cd liion-soc-ekf
!pip install -q -r requirements.txt
print('ready')


## 1. Check the model and estimators against closed-form theory

Eleven checks: OCV/derivative self-consistency, monotonicity, the plateau-vs-knee
slope ratio, the coulombic-efficiency asymmetry, EKF voltage tracking, the matched-model
RMSE bound, convergence timing, and the two mismatch-sensitivity checks (Coulomb
counting is exactly invariant to R0/OCV errors; the EKF is not).

Takes a few seconds.


In [ ]:
!python3 verify.py


## 2. Run the three experiments live

- **Matched initial condition**: both estimators start from the true SOC and the
  EKF's internal model exactly matches the plant. This isolates disturbance
  rejection from the 35 mA current-sensor bias.
- **Convergence from a bad guess**: the estimator starts several points away
  from the true SOC. This isolates recovery speed, not disturbance rejection.
- **Model mismatch**: the EKF's internal model gets a capacity error, an R0
  error, an OCV bias, or all three, while the plant keeps its true parameters.

Try changing `initial_est_soc` in the first call below and re-running to see how
little it moves the matched-init numbers, then compare against the convergence
study a few cells down.


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
from src.experiment import run_experiment
from src import robustness

data = run_experiment(initial_est_soc=0.82)   # <-- change me, e.g. 0.70
reference = data['true_soc']

for name, estimate in (('Coulomb Counting', data['cc_soc']),
                        ('Extended Kalman Filter', data['ekf_soc'])):
    err = (estimate - reference) * 100.0
    print(f'{name:24s} RMSE {np.sqrt(np.mean(err**2)):.3f} pp   '
          f'MAE {np.mean(np.abs(err)):.3f} pp   max {np.max(np.abs(err)):.3f} pp')


### Convergence from a bad initial guess

This is where the EKF's advantage actually shows up: it uses the voltage
measurement to correct a wrong starting SOC, while Coulomb counting can only
carry the initial error forward forever.


In [ ]:
conv, _ = robustness.convergence_study()
for name, r in conv.items():
    settle = 'never' if r['settle_time_s'] is None else f"{r['settle_time_s']:.0f} s"
    print(f"{name:24s} initial error {r['initial_error_pp']:.2f} pp   settled in {settle}")


### Model mismatch

Coulomb counting never touches R0, Rp, Cp, or the OCV table, so it is exactly
unaffected by errors in them (it is not immune to a bad capacity, since it
uses that directly to scale the integrated current). The EKF depends on all of
them to interpret the voltage measurement, so it is more sensitive to R0 and
OCV errors than to a capacity error alone.


In [ ]:
rows = robustness.mismatch_study()
print(f'{"scenario":24s} {"CC RMSE (pp)":>14s} {"EKF RMSE (pp)":>14s}')
for label, cc_rmse, ekf_rmse in rows:
    print(f'{label:24s} {cc_rmse:14.3f} {ekf_rmse:14.3f}')


## 3. Plot the matched-init tracking

Full 2-hour run, true SOC vs. both estimators.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4.5))
t_h = data['t'] / 3600.0
ax.plot(t_h, reference * 100.0, lw=1.8, color='black', label='True SOC')
ax.plot(t_h, data['cc_soc'] * 100.0, lw=1.3, label='Coulomb counting')
ax.plot(t_h, data['ekf_soc'] * 100.0, lw=1.3, label='EKF')
ax.set_xlabel('time (h)'); ax.set_ylabel('SOC (%)')
ax.set_title('Matched-initial-condition tracking'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()


## 4. Rebuild every figure in the README

About ten seconds.


In [ ]:
!python3 make_figures.py


In [ ]:
from IPython.display import SVG, display
for name in ['soc_comparison', 'convergence', 'voltage_fit', 'estimation_error']:
    display(SVG(filename=f'results/{name}_light.svg'))


---

Built by Fatin Nihal Islam. The full write-up, including what the model
deliberately leaves out, is in the
[repository README](https://github.com/fatinnihal532-hub/liion-soc-ekf).
